# 🎭 TalkMateAI 地端多模態 AI 伴侶 - Google Colab 雲端部署與執行手冊

本手冊是將 [TalkMetaAI GitHub 專案](https://github.com/wangsungwen/TalkMetaAI) 的部署步驟轉換為 Google Colab 環境的執行版本。我們將在此 Jupyter Notebook 中，帶您一步步在 Google Colab 的 Linux 雲端伺服器上安裝、編譯並運行 TalkMateAI 服務，並透過 `ngrok` 提供對外的安全 HTTPS 網址，讓您的手機（iOS/Android）或平板可以直接連線進行 3D AI 伴侶的實時語音與相機視覺對話！

## ✨ 專案核心特色：
- 🧠 **雙模式大腦支援**：多模態視覺版與繁體中文 Qwen2.5-1.5B 文本增強版。
- 🎙️ **實時語音與對嘴 (STT & Visemes)**：基於 Whisper-tiny 語音識別與 Kokoro (台灣腔女聲 `zs_jessica` / `zf_xiaoxiao`)，提供毫秒級嘴型同步時間戳記。
- 👁️ **WebRTC 實時視覺眼睛**：相機畫面定時擷取並轉換為 Base64 傳送至後端進行視覺脈絡互動。
- 🔌 **單埠一體化託管**：前端 Next.js 靜態打包，由後端 FastAPI 伺服器掛載託管在相同埠口，完美解決跨網域 CORS 問題與 ngrok 單通道限制。
- 🚀 **Colab GPU 滿血加速**：特別針對 Colab 的 T4 等 GPU 進行加速補丁，讓 Whisper、Qwen 模型運行在 GPU 上，實現毫秒級對答反應！

---  
## 🛠️ 步驟 1：檢測執行環境與啟用 GPU 加速

在開始之前，讓我們先檢查 Google Colab 是否已為此執行階段分配了 GPU (如 Tesla T4)。

**👉 提示：**
若 `CUDA 是否可用` 顯示 `False`，建議在 Colab 上方選單選擇 **「執行階段」 > 「變更執行階段類型」**，並在硬體加速器中選擇 **GPU (T4)**，這能讓 AI 大腦與語音識別的反應速度提升十倍以上！

In [ ]:
import torch
print("=========================================")
cuda_available = torch.cuda.is_available()
print(f"CUDA (GPU) 是否可用: {cuda_available}")
if cuda_available:
    print(f"GPU 設備型號: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ 警告：目前使用的是 CPU 模式。")
    print("建議在功能表選擇『執行階段』>『變更執行階段類型』，將硬體加速器變更為 GPU (如 T4) 以獲得最佳對話流暢度！")
print("=========================================")

---  
## 📥 步驟 2：複製專案程式碼與定位目錄

我們使用 `git clone` 將 TalkMetaAI 的完整專案程式碼複製到 Colab 虛擬機的 `/content` 目錄中，並將工作路徑切換至專案根目錄。

In [ ]:
# 複製 TalkMetaAI 專案代碼
!git clone https://github.com/wangsungwen/TalkMetaAI.git

# 切換至專案根目錄 (採用絕對路徑，防範目錄定位錯誤)
%cd /content/TalkMetaAI

---  
## 📦 步驟 3：安裝前端開發與套件管理工具 (Node.js & PNPM)

TalkMateAI 的前端是基於 Next.js 15 與 TypeScript 寫成，因此需要 Node.js 運行環境。
為了快速且節省空間地安裝前端套件，本專案採用 `pnpm` 套件管理器。Colab 系統中通常預裝了 Node.js，我們僅需要使用 `npm` 全域安裝 `pnpm` 即可。

In [ ]:
# 檢查 Node.js 版本 (建議 >= 20.x)
!node -v

# 使用 npm 全域安裝 pnpm 套件管理器
!npm install -g pnpm

---  
## 🔧 步驟 4：設定前端靜態打包參數與編譯前端

為了在同一個連接埠 (Port 8000) 下同時執行前端網頁與後端 API，避免跨網域 (CORS) 阻擋與方便 ngrok 對外映射，我們需要讓 Next.js 前端以**純靜態網頁 HTML/CSS/JS** 形式匯出 (`output: 'export'`)。

這一步驟我們使用 Python 檢查設定檔。因為專案中可能同時包含 `next.config.js`、`next.config.mjs` 或 `next.config.ts`，**若多重檔案共存會導致 Next.js 打包衝突失敗並出現 404 錯誤**。
我們將使用 Python 優先確保 `next.config.ts` 具有 `output: 'export'`，並刪除其他衝突檔案以確保 `pnpm run build` 打包出靜態檔案至 `apps/client/out`。

In [ ]:
import os

client_dir = "/content/TalkMetaAI/apps/client"
config_js = os.path.join(client_dir, "next.config.js")
config_ts = os.path.join(client_dir, "next.config.ts")
config_mjs = os.path.join(client_dir, "next.config.mjs")

# 確保 client 目錄存在 (防範使用者跳步驟執行)
if not os.path.exists(client_dir):
    print("❌ 錯誤：找不到前端目錄，請確認步驟 2 已順利 clone 專案！")
else:
    # 優先使用專案自帶的 next.config.ts
    if os.path.exists(config_ts):
        print("✓ 偵測到 next.config.ts，將使用其進行靜態導出設定")
        with open(config_ts, 'r', encoding='utf-8') as f:
            content = f.read()
        if "output" not in content or "export" not in content:
            new_content = """/** @type {import('next').NextConfig} */
const nextConfig = {
  output: 'export',
  images: {
    unoptimized: true,
  },
  typescript: {
    ignoreBuildErrors: true,
  },
  eslint: {
    ignoreDuringBuilds: true,
  },
};

module.exports = nextConfig;
"""
            with open(config_ts, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print("✓ next.config.ts 已成功寫入靜態打包設定")
        
        # 移除衝突的 js / mjs 檔案
        if os.path.exists(config_js): os.remove(config_js)
        if os.path.exists(config_mjs): os.remove(config_mjs)
    else:
        print("🔧 建立 next.config.js 進行靜態導出設定...")
        next_config_content = """/** @type {import('next').NextConfig} */
const nextConfig = { 
  output: 'export',
  images: {
    unoptimized: true,
  },
  typescript: {
    ignoreBuildErrors: true,
  },
  eslint: {
    ignoreDuringBuilds: true,
  };

module.exports = nextConfig;
"""
        with open(config_js, "w", encoding="utf-8") as f:
            f.write(next_config_content)
        if os.path.exists(config_mjs): os.remove(config_mjs)
        print("✓ next.config.js 設定完成")

In [ ]:
# 切換到前端 client 目錄 (使用絕對路徑)
%cd /content/TalkMetaAI/apps/client

# 強制清除之前的建置快取，避免殘留檔案佔用
!rm -rf .next out

# 授權前端 pnpm 執行建置腳本，並安裝前端相依套件
!pnpm approve-builds || true
!pnpm install

# 執行 Next.js 打包建置 (成品將會輸出至 apps/client/out)
!pnpm run build

# 返回專案根目錄 (使用絕對路徑)
%cd /content/TalkMetaAI

---  
## 🐍 步驟 5：安裝 Python 後端與 AI 語音中文依賴套件

後端使用 FastAPI 開發，並整合了 `openai/whisper-tiny` (語音辨識 STT)、`Qwen/Qwen2.5-1.5B-Instruct` (語言模型大腦) 以及 `Kokoro` (語音合成 TTS)。

為了讓語音管線正常處理繁體中文語音分詞、注音轉換與拼音，我們必須額外安裝 `jieba`、`g2pM`、`ordered-set`、`pypinyin` 與 `cn2an`。在 Colab 中，我們將直接使用系統 pip 安裝這些後端 Python 依賴。

In [ ]:
# 安裝一體化主機 FastAPI、Uvicorn，以及 AI 推理與中文語音合成必備的 Python 套件
!pip install fastapi uvicorn pillow transformers websockets numpy kokoro jieba g2pM ordered-set pypinyin cn2an soundfile accelerate scipy

---  
## ⚡ 步驟 6：GPU 滿血加速修正補丁 (自動對齊 CUDA)

TalkMateAI 後端程式碼 `main.py` 預設將 AI 模型載入設備硬編碼為 `"cpu"`。為了在 Colab 提供的 GPU 上發揮硬體「滿血加速」實力，我們在此處使用 Python 對 `main.py` 進行動態修改補丁。

**⚠️ 說明**：為了防範使用者在先前的步驟中切換了工作路徑（例如切換到 `apps/server` 啟動服務），**本單元格採用絕對路徑讀取與寫入 `main.py`**，從根本上避免出現 `FileNotFoundError` 錯誤。

### 補丁修改事項：
1. **加入全域 `import torch`**：避免執行階段出現 `NameError: name 'torch' is not defined` 的錯誤。
2. **載入裝置自動修改**：將 Whisper 語音辨識、Qwen 大腦模型與 Kokoro 語音合成的載入裝置修改為 `"cuda" if torch.cuda.is_available() else "cpu"`。
3. **權重精度自動轉換**：當使用 GPU 時，自動將 Whisper 與 Qwen 精度轉換為 `float16`（半精度），大幅降低顯存佔用並倍增推理速度！

In [ ]:
import re
import os

# 採用絕對路徑，確保無論目前在哪個資料夾下，皆能成功修改
main_py_path = "/content/TalkMetaAI/apps/server/main.py"

if not os.path.exists(main_py_path):
    print(f"❌ 錯誤：找不到後端 main.py，請確認您的專案路徑是否為 '/content/TalkMetaAI'！")
else:
    with open(main_py_path, "r", encoding="utf-8") as f:
        content = f.read()

    # 補丁 0：在 main.py 最頂部加入 import torch 確保全域可用，徹底修復 NameError 錯誤
    # 注意：不可只檢查 "import torch" 是否在 content 中，因為局部函式內本來就有該字串。
    # 我們必須精確地在檔案最開頭加入，且確保不重複加入。
    if not content.startswith("import torch\n"):
        content = "import torch\n" + content

    # 補丁 1：修改 WhisperProcessor，支援 GPU / CUDA 及 float16 半精度
    content = re.sub(
        r'class WhisperProcessor:.*?def __init__\(self\):.*?self\.device = "cpu"',
        lambda m: m.group(0).replace('self.device = "cpu"', 'self.device = "cuda" if torch.cuda.is_available() else "cpu"'),
        content,
        flags=re.DOTALL
)
    content = re.sub(
        r'class WhisperProcessor:.*?def __init__\(self\):.*?self\.torch_dtype = torch\.float32',
        lambda m: m.group(0).replace('self.torch_dtype = torch.float32', 'self.torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32'),
        content,
        flags=re.DOTALL
)

    # 補丁 2：修改 QwenLLMProcessor，支援 GPU / CUDA 及 float16 半精度
    content = re.sub(
        r'class QwenLLMProcessor:.*?def __init__\(self\):.*?self\.device = "cpu"',
        lambda m: m.group(0).replace('self.device = "cpu"', 'self.device = "cuda" if torch.cuda.is_available() else "cpu"'),
        content,
        flags=re.DOTALL
)
    content = re.sub(
        r'class QwenLLMProcessor:.*?def __init__\(self\):.*?self\.torch_dtype = torch\.float32',
        lambda m: m.group(0).replace('self.torch_dtype = torch.float32', 'self.torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32'),
        content,
        flags=re.DOTALL
)

    # 補丁 3：修改 KokoroTTSProcessor，支援 GPU / CUDA 載入
    content = re.sub(
        r'class KokoroTTSProcessor:.*?def __init__\(self\):.*?self\.device = "cpu"',
        lambda m: m.group(0).replace('self.device = "cpu"', 'self.device = "cuda" if torch.cuda.is_available() else "cpu"'),
        content,
        flags=re.DOTALL
)

    # 將修改寫回 main.py
    with open(main_py_path, "w", encoding="utf-8") as f:
        f.write(content)

    print("✓ GPU 滿血加速修正補丁套用成功！Whisper, Qwen, Kokoro 將會在 GPU 上運行，且已補齊 torch 全域宣告。")

---  
## 🌐 步驟 7：配置 ngrok 公網穿透

由於行動裝置瀏覽器（如手機上的 Safari 或 Chrome）的隱私與安全性政策，**麥克風與相機鏡頭權限僅允許在安全連接 (HTTPS) 下調用**。

因為 Google Colab 位於雲端，其本地 IP `127.0.0.1` 無法直接從您的手機或外部瀏覽器連線。我們必須使用 `ngrok` 工具建立一條安全且免費的 HTTPS 隧道，把 Colab 虛擬機中的 Port 8000 對外映射出來。

### 🔑 如何取得 ngrok Authtoken：
1. 請前往 [ngrok 官網 (ngrok.com)](https://ngrok.com/) 免費註冊並登入帳號。
2. 在左側儀表板中點選 **Your Authtoken**。
3. 複製您的 Authtoken 密鑰。
4. 貼在下方代碼的 `NGROK_TOKEN` 欄位中，然後執行該單元格。

In [ ]:
#@title 🔑 輸入您的 ngrok Authtoken
NGROK_TOKEN = "" #@param {type:"string"}

import os
if not NGROK_TOKEN:
    print("❌ 錯誤：請先在上方輸入框填入您的 ngrok Authtoken 密鑰，再執行本單元格！")
else:
    # 下載 Linux 版 ngrok
    if not os.path.exists("ngrok"):
        print("📥 正在下載 ngrok 壓縮包...")
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
        print("📦 正在解壓縮 ngrok...")
        !tar -xzf ngrok-v3-stable-linux-amd64.tgz
        
    # 寫入 token 至設定
    !./ngrok config add-authtoken {NGROK_TOKEN}
    print("✓ ngrok 設定完成！您可以進行下一步啟動一體化服務。")

---  
## 🚀 步驟 8：啟動 ngrok 穿透並開啟 FastAPI 服務

這一步驟我們將在背景啟動 ngrok 穿透 Port 8000，並透過 API 獲取對應的 HTTPS 公網網址。
隨後，我們將切換到 `apps/server` 目錄啟動 Uvicorn FastAPI 服務。

**👉 使用方式：**
1. 執行本單元格，等候數秒。
2. 您將會看見輸出印出 **`Public URL: https://xxxx.ngrok-free.app`** 的專屬網址。
3. **點擊該網址** 即可開啟 3D 伴侶網頁！

**⚠️ 溫馨提醒：**
- 進行語音對話測試時，**強烈建議配戴耳機**！若直接用喇叭放音，聲音會再次被麥克風錄入造成回音迴圈，導致 AI 不斷自言自語或判定您一直在說話。
- **請勿將網址直接點開在 LINE 等通訊軟體內建瀏覽器**，因為這些瀏覽器會阻擋相機/麥克風權限。請複製網址改用手機系統預設的 **Safari** 或 **Chrome** 瀏覽器開啟。

In [ ]:
import subprocess
import time
import urllib.request
import json
import os

if not os.path.exists("ngrok"):
    print("❌ 錯誤：偵測不到 ngrok 執行檔，請確保步驟 7 已順利執行並填寫 Token！")
else:
    # 清理舊的 ngrok 行程
    subprocess.run(["pkill", "-f", "ngrok"])
    
    print("📡 正在背景建立安全 HTTPS 隧道 (Port 8000)... ")
    ngrok_process = subprocess.Popen(["./ngrok", "http", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4) # 等待網道啟動建立
    
    try:
        # 向 ngrok 的本地管理介面查詢公網 URL
        res = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=3)
        tunnel_info = json.loads(res.read().decode())
        public_url = tunnel_info['tunnels'][0]['public_url']
        print("\n" + "="*75)
        print(f"🎉 TalkMateAI 已經成功在 Colab 啟動並對外穿透！")
        print(f"🔗 請點擊以下公網 HTTPS 連結在手機或電腦端開啟 3D 伴侶：")
        print(f"\n👉 {public_url} 👈\n")
        print("="*75 + "\n")
    except Exception as e:
        print("⚠️ 無法取得 ngrok 公網連結，請確保 Token 正確且 ngrok 沒有殘留背景行程！")

# 啟動 FastAPI 整合託管後端 (採用絕對路徑防止定位衝突)
print("🔥 正在啟動後端伺服器並載入大腦模型 (這通常需要 1 到 2 分鐘，請載入完成後連線)...\n")
%cd /content/TalkMetaAI/apps/server
!uvicorn main:app --host 0.0.0.0 --port 8000

---  
## ☁️ 延伸閱讀：從 GitHub 部署至 Hugging Face Space 雲端

如果您不想每次都開啟 Colab 並架設 ngrok，您可以將 TalkMateAI 部署成 **Hugging Face Docker Space**，讓不同網路環境的手機、平板直接透過 HTTPS 存取，無須自己處理本機 HTTPS、CORS、公網穿透與防火牆設定。

### 1. 建立 Hugging Face Space
1. 前往 [Hugging Face Space 建立頁面](https://huggingface.co/new-space)。
2. **Owner** 選擇您的 Hugging Face 帳號，**Space name** 填入 `TalkmetaAI`。
3. **SDK** 選擇 `Docker`。
4. **Hardware** 先選擇 `CPU Basic` 即可啟動服務（免費）；需要更快回應速度時再付費升級為 GPU。

### 2. 確認設定檔設定
- 確保根目錄 `README.md` 的 YAML metadata 包含：
  ```yaml
  sdk: docker
  app_port: 7860
  ```
- 確保 `Dockerfile` 中最後啟動 FastAPI 時監聽 `0.0.0.0` 與指定 Port：
  ```dockerfile
  EXPOSE 7860
  CMD sh -c "uvicorn main:app --host 0.0.0.0 --port ${PORT:-7860}"
  ```
- 確保 `Dockerfile` 已補齊中文分詞與拼音依賴套件（如 `cn2an`, `g2pM`, `jieba`, `kokoro`, `numpy`, `ordered-set`, `pypinyin` 等）。

### 3. 推送與同步
將程式碼推送至 GitHub 後，如果您的 Space 有連結 GitHub 倉庫，它會自動進行 Rebuild 部署。您也可以手動添加 Hugging Face Space 作為 git remote 並推送：
```bash
git remote add hf https://huggingface.co/spaces/您的使用者名稱/TalkmetaAI
git push hf main:main
```

---  
## 🔍 進階故障排除 (Troubleshooting)

| 故障現象 (Symptom) | 核心原因剖析 (Root Cause) | 技術對策與除錯指令 (Resolution) |
| :--- | :--- | :--- |
| **開口說話毫無反應**，前端 VAD 顯示傳輸中，但後端沒有 any 接收與處理日誌。 | 前端語音框架發送的二進位欄位名稱 (Key) 與後端 `main.py` 條件判斷不對稱，導致語音封包被忽略。 | 1. 檢查前端發送的二進位 Key 與後端 `main.py` 是否對齊。<br>2. 確保使用最新版支援多重二進位格式（Raw Bytes/Blob/JSON）接收機制的 `main.py`。 |
| **後端狂噴長音訊異常** `Transcription error... > 30 seconds` | 使用者未戴耳機，喇叭播出的 AI 語音被麥克風重複錄入，形成回音環路，造成 VAD 判定無限說話。 | 1. **鐵律**：對話測試時必須配戴耳機，物理性切斷回音環路。<br>2. 在網頁點開 Detection Settings，將 `Silence Duration`（靜音判定時長）調敏感至 `800ms`，讓錄音一停頓即立刻切斷並送出。 |
| **網頁控制台報錯 WebSocket error** | 1. 後端 FastAPI 服務未成功開啟。<br>2. 連結未經由 ngrok HTTPS 代理。<br>3. 瀏覽器阻擋非 HTTPS 呼叫麥克風。 | 1. 檢查後端執行視窗是否順利跑出 `Application startup complete.` 等就緒提示。<br>2. 請務必使用步驟 8 產生的 `https` 連結（而非 Colab 內部 `127.0.0.1`），如此瀏覽器方能順利向 WebSocket 提供鏡頭與錄音權限。 |